# 07.5 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/05-ml-forecasting/exercises.ipynb)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

toy = pd.Series([30, 32, 28, 40, 25, 0, 18, 31, 33, 29, 42, 26],     # £k, starting on a Monday
                index=pd.date_range("2011-11-07", periods=12, freq="D"))

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

from sklearn.ensemble import HistGradientBoostingRegressor

def daily_table(d):
    """Turn the daily series into a table. Every lag is >= 35 days, so 5 weeks ahead is honest."""
    t = pd.DataFrame({"y": d})
    t["lag_35"] = d.shift(35)
    t["mean_28_lag_35"] = d.shift(35).rolling(28).mean()
    t["dow"] = d.index.day_of_week
    t["month"] = d.index.month
    t["day_of_year"] = d.index.day_of_year
    t["days_to_xmas"] = (pd.to_datetime(d.index.year.astype(str) + "-12-25") - d.index).days
    return t

TABLE = daily_table(daily)
FEATURES = [c for c in TABLE.columns if c != "y"]

def gbm(train, h):
    """Train on daily rows up to the origin, predict the next h weeks of days, sum to weeks."""
    origin = train.index[-1]
    past = TABLE[TABLE.index <= origin].dropna()
    future = TABLE[TABLE.index > origin].iloc[: 7 * h]
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=10, random_state=0)
    model.fit(past[FEATURES], past["y"])
    pred = pd.Series(model.predict(future[FEATURES]), index=future.index).clip(lower=0)
    pred[future["dow"] == 5] = 0                       # ShopWave never trades on Saturdays
    return pred.resample("W").sum().values[:h]

## Exercise 1: by hand

Using `toy`, build by hand the row for **Saturday 12 November** with columns `lag_1`, `lag_7`, `mean_3` (the 3 days before), `dow`. Then check with Pandas.

In [ ]:
# your code here

## Exercise 2: fewer features

Remove `days_to_xmas` and `day_of_year` from `FEATURES` and re-run the backtest of `gbm`. How much worse is it? What does that tell you about where the model's skill comes from?

In [ ]:
# your code here

## Exercise 3: the weekly table

Build a **weekly** table instead of daily: features `lag_5` (5 weeks ago), `lag_52` (same week last year), `week_of_year`. Train gradient boosting and backtest it. How does it compare with the daily model and with seasonal naive?

In [ ]:
# your code here

## Exercise 4: quantiles for safety stock

`HistGradientBoostingRegressor(loss="quantile", quantile=0.9)` predicts the 90th percentile instead of the average. Train one on the daily table up to 30 October 2011 and forecast 6 November – 4 December. What fraction of actual weeks were below the 90% forecast?

In [ ]:
# your code here